# LeetCode 1813 — Sentence Similarity III bằng Python

Link đề: https://leetcode.com/problems/sentence-similarity-iii/ · Độ khó: **Medium** · Chủ đề: Array, Two Pointers, String

Skeleton LeetCode:

```python
class Solution:
    def areSentencesSimilar(self, sentence1: str, sentence2: str) -> bool:
        ...
```

Lộ trình: **thử xóa mọi khối từ liên tiếp** khỏi câu dài (baseline) → **thử mọi điểm cắt** của câu ngắn thành tiền tố + hậu tố → **hai con trỏ: khớp tiền tố chung + hậu tố chung** `O(m + n)` (lời giải nộp) → **`deque` bóc hai đầu**.

## 1. Mục tiêu học

1. Làm việc ở mức **từ** (`split()`), không ở mức ký tự — tránh bẫy `"Frog"` vs `"Frogs"`.
2. Diễn đạt lại "chèn một câu vào giữa" thành: câu ngắn = **tiền tố** của câu dài + **hậu tố** của câu dài.
3. Dùng **hai con trỏ** đếm độ dài tiền tố chung `i` và hậu tố chung `j`, kết luận bằng `i + j ≥ n`.
4. Hiểu vì sao cần **đổi chỗ** để câu thứ hai luôn là câu ngắn hơn, và vì sao `i + j` có thể **vượt** `n` (khi có từ lặp).
5. Kiểm chứng bằng vét cạn "xóa một khối" trên câu ngẫu nhiên có seed.

## 2. Đề bài & ví dụ

Cho hai chuỗi `sentence1`, `sentence2`, mỗi chuỗi là một **câu**: danh sách các **từ** cách nhau đúng một dấu cách, không có cách thừa ở đầu/cuối. Mỗi từ chỉ gồm chữ cái tiếng Anh hoa/thường.

Hai câu `s1`, `s2` **tương tự** nếu có thể **chèn một câu bất kỳ (có thể rỗng)** vào **bên trong một** trong hai câu để hai câu trở nên bằng nhau. Câu được chèn phải được ngăn cách với các từ có sẵn bằng dấu cách.

- `"Hello Jane"` và `"Hello my name is Jane"` tương tự (chèn `"my name is"` giữa `Hello` và `Jane`).
- `"Frog cool"` và `"Frogs are cool"` **không** tương tự: dù chèn `"s are"` vào thì `"s"` không được ngăn cách với `"Frog"` bằng dấu cách.

Trả về `true` nếu hai câu tương tự, ngược lại `false`.

**Ví dụ 1:** `"My name is Haley"`, `"My Haley"` → `true` (chèn `"name is"`).

**Ví dụ 2:** `"of"`, `"A lot of words"` → `false`.

**Ví dụ 3:** `"Eating right now"`, `"Eating"` → `true` (chèn `"right now"` ở cuối).

**Ràng buộc:** `1 ≤ sentence1.length, sentence2.length ≤ 100`; chỉ gồm chữ cái và dấu cách; các từ cách nhau đúng một dấu cách.

In [1]:
import random
import time
from collections import deque
from typing import List

examples = [
    ("My name is Haley", "My Haley", True),
    ("of", "A lot of words", False),
    ("Eating right now", "Eating", True),
    ("Hello Jane", "Hello my name is Jane", True),      # ví dụ trong phần mô tả
    ("Frog cool", "Frogs are cool", False),             # ví dụ trong phần mô tả
]
for a, b, want in examples:
    print(f"{a!r:22} | {b!r:26} -> {want}")

'My name is Haley'     | 'My Haley'                 -> True
'of'                   | 'A lot of words'           -> False
'Eating right now'     | 'Eating'                   -> True
'Hello Jane'           | 'Hello my name is Jane'    -> True
'Frog cool'            | 'Frogs are cool'           -> False


## 3. Kiến thức nền

### 3.1 `str.split()` tách câu thành list từ

In [2]:
print("My name is Haley".split())
print("Frogs are cool".split(), "| số từ:", len("Frogs are cool".split()))

['My', 'name', 'is', 'Haley']
['Frogs', 'are', 'cool'] | số từ: 3


### 3.2 So sánh list theo lát cắt

`a[:k] == b[:k]` so `k` phần tử đầu; `a[len(a)-k:] == b[len(b)-k:]` so `k` phần tử cuối. Chú ý `a[-0:]` là **cả list** (vì `-0 == 0`), nên với `k = 0` phải viết `a[len(a)-k:]` chứ không phải `a[-k:]`.

In [3]:
a = ["My", "name", "is", "Haley"]
print(a[:1], a[len(a) - 1:])
print("a[-0:] =", a[-0:], "| a[len(a)-0:] =", a[len(a) - 0:])

['My'] ['Haley']
a[-0:] = ['My', 'name', 'is', 'Haley'] | a[len(a)-0:] = []


### 3.3 Hai con trỏ đi từ hai đầu

Một con trỏ `i` tiến từ đầu khi hai list còn khớp; một con trỏ `j` tiến từ cuối (so `a[m-1-j]` với `b[n-1-j]`).

In [4]:
a, b = ["x", "y", "z", "w"], ["x", "w"]
i = 0
while i < len(b) and a[i] == b[i]:
    i += 1
j = 0
while j < len(b) and a[-1 - j] == b[-1 - j]:
    j += 1
print("tiền tố chung:", i, "| hậu tố chung:", j)

tiền tố chung: 1 | hậu tố chung: 1


### 3.4 Đổi chỗ để chuẩn hóa

`if len(a) < len(b): a, b = b, a` — sau dòng này `a` luôn là câu dài hơn (hoặc bằng). Giảm một nửa số trường hợp phải nghĩ.

In [5]:
a, b = ["Eating"], ["Eating", "right", "now"]
if len(a) < len(b):
    a, b = b, a
print("dài:", a, "| ngắn:", b)

dài: ['Eating', 'right', 'now'] | ngắn: ['Eating']


### 3.5 `collections.deque`

Hàng đợi hai đầu: `popleft()` và `pop()` đều `O(1)` — tiện để "bóc" từ khớp ở hai đầu.

In [6]:
dq = deque(["A", "lot", "of", "words"])
print(dq.popleft(), dq.pop(), list(dq))

A words ['lot', 'of']


## 4. Ý tưởng nền tảng / trực giác

Chèn một câu vào câu **ngắn** `b` để được câu **dài** `a` ⇔ `a` = (một **tiền tố** của `b`) + (đoạn chèn) + (phần **còn lại** của `b`). Nói cách khác: cắt `b` thành `b[:k]` và `b[k:]`, thì `b[:k]` phải là **tiền tố** của `a` và `b[k:]` phải là **hậu tố** của `a`.

(Chèn vào câu dài để bằng câu ngắn là không thể — câu chỉ dài thêm. Nếu hai câu cùng độ dài thì phải bằng nhau, đoạn chèn rỗng.)

Tính tay Ví dụ 1: `a = [My, name, is, Haley]`, `b = [My, Haley]`.
- Tiền tố chung: `My` = `My` ✔, `name` ≠ `Haley` ✘ ⇒ `i = 1`.
- Hậu tố chung: `Haley` = `Haley` ✔, `is` ≠ `My` ✘ ⇒ `j = 1`.
- `i + j = 2 ≥ len(b) = 2` ⇒ chọn `k = 1`: `b[:1] = [My]` là tiền tố, `b[1:] = [Haley]` là hậu tố ✔ ⇒ `true`.

Ví dụ 2: `a = [A, lot, of, words]`, `b = [of]`: `i = 0` (`A ≠ of`), `j = 0` (`words ≠ of`) ⇒ `0 < 1` ⇒ `false`. Từ `of` có mặt trong `a` nhưng nằm **giữa** — muốn ra `a` phải chèn ở **cả hai** phía.

## 5. Cách 1 — Brute Force / baseline: thử xóa mọi khối từ liên tiếp

**Ý tưởng:** "chèn đoạn `X` vào `b` để được `a`" ⇔ "xóa một khối liên tiếp (có thể rỗng) khỏi `a` để được `b`". Thử mọi khối `a[l:r]` (`0 ≤ l ≤ r ≤ m`), xem `a[:l] + a[r:] == b`. Làm cho cả hai chiều để khỏi phải đổi chỗ.

In [7]:
def similar_brute(s1, s2):
    a, b = s1.split(), s2.split()
    def can_delete_block(x, y):
        m = len(x)
        for l in range(m + 1):
            for r in range(l, m + 1):
                if x[:l] + x[r:] == y:
                    return True
        return False
    return can_delete_block(a, b) or can_delete_block(b, a)

for a, b, want in examples:
    print(similar_brute(a, b), "| kỳ vọng", want)

True | kỳ vọng True
False | kỳ vọng False
True | kỳ vọng True
True | kỳ vọng True
False | kỳ vọng False


**Độ phức tạp:** `O(m²)` khối × `O(m)` để ghép và so ⇒ `O(m³)` với `m` = số từ câu dài (≤ 50 vì độ dài chuỗi ≤ 100). Ràng buộc nhỏ nên vẫn chạy được, nhưng ta đang thử rất nhiều khối vô ích: độ dài khối **bắt buộc** là `m − n`.

**Nút thắt:** chỉ cần chọn điểm cắt `l`; `r = l + (m − n)` được xác định luôn.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1:** chỉ có `n + 1` cách cắt `b` (điểm cắt `k = 0..n`) ⇒ thử từng `k`: `a[:k] == b[:k]` và `a[m-(n-k):] == b[k:]`.

**Insight 2 (đơn điệu):** điều kiện "`b[:k]` là tiền tố của `a`" đúng với mọi `k ≤ i` và sai với `k > i`, trong đó `i` = **độ dài tiền tố chung dài nhất**. Tương tự "`b[k:]` là hậu tố của `a`" đúng ⇔ `n − k ≤ j` với `j` = độ dài hậu tố chung dài nhất. Tồn tại `k` thỏa cả hai ⇔ tồn tại `k` với `n − j ≤ k ≤ i` ⇔ **`i + j ≥ n`**.

**Quy tắc:** đổi chỗ để `n = len(b) ≤ len(a) = m`; đếm `i` (tiến từ đầu, `i < n`), đếm `j` (tiến từ cuối, `j < n`); trả `i + j ≥ n`.

Chú ý `i` và `j` có thể **chồng nhau** (ví dụ `a = [A, A, A]`, `b = [A, A]` cho `i = j = 2`) — không sao, điều kiện là `≥` chứ không phải `==`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Thử mọi điểm cắt của câu ngắn (`O(n · m)`)

In [8]:
def similar_cuts(s1, s2, verbose=False):
    a, b = s1.split(), s2.split()
    if len(a) < len(b):
        a, b = b, a
    m, n = len(a), len(b)
    for k in range(n + 1):
        ok_pre = a[:k] == b[:k]
        ok_suf = a[m - (n - k):] == b[k:]
        if verbose:
            print(f"k={k}: tiền tố {b[:k]} {'✔' if ok_pre else '✘'} | hậu tố {b[k:]} {'✔' if ok_suf else '✘'}")
        if ok_pre and ok_suf:
            return True
    return False

print("=>", similar_cuts("My name is Haley", "My Haley", verbose=True))

k=0: tiền tố [] ✔ | hậu tố ['My', 'Haley'] ✘
k=1: tiền tố ['My'] ✔ | hậu tố ['Haley'] ✔
=> True


**Độ phức tạp:** `n + 1` điểm cắt × `O(n)` so sánh ⇒ `O(n²)` (cộng `O(m)` để tách). Nút thắt: so tiền tố/hậu tố lại từ đầu cho mỗi `k`; nhưng theo Insight 2 chỉ cần biết hai số `i`, `j`.

### 7.2 Cách 3 — Hai con trỏ: tiền tố chung + hậu tố chung (`O(m + n)`) — lời giải chuẩn

In [9]:
class SolutionTwoPointers:
    def areSentencesSimilar(self, sentence1: str, sentence2: str) -> bool:
        a, b = sentence1.split(), sentence2.split()
        if len(a) < len(b):
            a, b = b, a
        m, n = len(a), len(b)
        i = 0
        while i < n and a[i] == b[i]:
            i += 1
        j = 0
        while j < n and a[m - 1 - j] == b[n - 1 - j]:
            j += 1
        return i + j >= n

for a, b, want in examples:
    print(SolutionTwoPointers().areSentencesSimilar(a, b) == want)

True
True
True
True
True


**Độ phức tạp:** tách câu `O(L₁ + L₂)` (độ dài chuỗi), hai vòng `while` `O(n)` ⇒ `O(L₁ + L₂)`; bộ nhớ `O(m + n)` cho hai list từ.

### 7.3 Cách 4 — `deque`: bóc từ khớp ở hai đầu

Bóc trái khi hai từ đầu bằng nhau, bóc phải khi hai từ cuối bằng nhau; tương tự nếu **câu ngắn** bị bóc hết thì tương tự. Cần cẩn thận: bóc trái trước "hết mức", rồi mới bóc phải.

In [10]:
class SolutionDeque:
    def areSentencesSimilar(self, sentence1: str, sentence2: str) -> bool:
        a, b = deque(sentence1.split()), deque(sentence2.split())
        while a and b and a[0] == b[0]:
            a.popleft(); b.popleft()
        while a and b and a[-1] == b[-1]:
            a.pop(); b.pop()
        return not a or not b

for a, b, want in examples:
    print(SolutionDeque().areSentencesSimilar(a, b) == want)

True
True
True
True
True


Vì sao không cần đổi chỗ? Sau khi bóc, nếu một trong hai rỗng thì câu rỗng đó là câu ngắn và phần còn lại của câu kia chính là đoạn cần chèn. Bóc "hết mức" từ trái tương đương `i` lớn nhất có thể (nhưng bị chặn bởi câu ngắn hơn), sau đó phần phải tương đương `j` trên phần còn lại — và vì `i + j ≥ n` là điều kiện "có điểm cắt", bóc tham lam không làm mất lời giải (xem mục 10).

**Độ phức tạp:** `O(m + n)`.

## 8. TRACE từng bước

Trace Cách 3 trên Ví dụ 2 và một ví dụ có từ lặp (tự dựng): `a = "A A A B"`, `b = "A B"`.

| bước | so sánh | kết quả |
|---|---|---|
| Ví dụ 2, `i` | `A` vs `of` | ✘ ⇒ `i = 0` |
| Ví dụ 2, `j` | `words` vs `of` | ✘ ⇒ `j = 0` |
| Ví dụ 2 | `0 + 0 ≥ 1`? | **false** |
| lặp, `i` | `A`=`A` ✔, `A` vs `B` ✘ | `i = 1` |
| lặp, `j` | `B`=`B` ✔, `A`=`A` ✔ | `j = 2` (đã hết `b`) |
| lặp | `1 + 2 ≥ 2`? | **true** (`i`, `j` chồng nhau) |

In [11]:
def trace_two_pointers(s1, s2):
    a, b = s1.split(), s2.split()
    if len(a) < len(b):
        a, b = b, a; print("đổi chỗ: câu dài =", a)
    m, n = len(a), len(b)
    i = 0
    while i < n and a[i] == b[i]:
        print(f"  tiền tố: a[{i}]={a[i]!r} == b[{i}]={b[i]!r}"); i += 1
    if i < n: print(f"  tiền tố dừng: a[{i}]={a[i]!r} != b[{i}]={b[i]!r}")
    j = 0
    while j < n and a[m - 1 - j] == b[n - 1 - j]:
        print(f"  hậu tố: a[{m-1-j}]={a[m-1-j]!r} == b[{n-1-j}]={b[n-1-j]!r}"); j += 1
    if j < n: print(f"  hậu tố dừng: a[{m-1-j}]={a[m-1-j]!r} != b[{n-1-j}]={b[n-1-j]!r}")
    print(f"  i={i}, j={j}, n={n} -> {i + j >= n}")
    return i + j >= n

trace_two_pointers("of", "A lot of words")
trace_two_pointers("A A A B", "A B")

đổi chỗ: câu dài = ['A', 'lot', 'of', 'words']
  tiền tố dừng: a[0]='A' != b[0]='of'
  hậu tố dừng: a[3]='words' != b[0]='of'
  i=0, j=0, n=1 -> False
  tiền tố: a[0]='A' == b[0]='A'
  tiền tố dừng: a[1]='A' != b[1]='B'
  hậu tố: a[3]='B' == b[1]='B'
  hậu tố: a[2]='A' == b[0]='A'
  i=1, j=2, n=2 -> True


True

In [12]:
trace_two_pointers("Frog cool", "Frogs are cool")

đổi chỗ: câu dài = ['Frogs', 'are', 'cool']
  tiền tố dừng: a[0]='Frogs' != b[0]='Frog'
  hậu tố: a[2]='cool' == b[1]='cool'
  hậu tố dừng: a[1]='are' != b[0]='Frog'
  i=0, j=1, n=2 -> False


False

## 9. Code nộp LeetCode

In [13]:
class Solution:
    def areSentencesSimilar(self, sentence1: str, sentence2: str) -> bool:
        a, b = sentence1.split(), sentence2.split()
        if len(a) < len(b):                 # a là câu dài, b là câu ngắn
            a, b = b, a
        m, n = len(a), len(b)
        i = 0                               # độ dài tiền tố chung
        while i < n and a[i] == b[i]:
            i += 1
        j = 0                               # độ dài hậu tố chung
        while j < n and a[m - 1 - j] == b[n - 1 - j]:
            j += 1
        return i + j >= n                   # có điểm cắt k với n-j <= k <= i

for a, b, want in examples:
    assert Solution().areSentencesSimilar(a, b) == want
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

Đặt `a` (dài `m`), `b` (dài `n ≤ m`) là list từ.

**Bổ đề 1:** hai câu tương tự ⇔ tồn tại `k ∈ [0, n]` sao cho `a[:k] = b[:k]` và `a[m-n+k:] = b[k:]`.
*(⇐)* chèn `a[k : m-n+k]` vào `b` ngay sau từ thứ `k` ⇒ được `a`. *(⇒)* chèn vào `b` (câu ngắn) được `a`: phần trước chỗ chèn là `b[:k] = a[:k]`, phần sau là `b[k:]` = hậu tố cùng độ dài của `a`. Chèn vào `a` chỉ làm `a` dài hơn `b`, nên chỉ có thể bằng `b` khi đoạn chèn rỗng và `m = n`, khi đó `a = b` — cũng thỏa với `k = n`.

**Bổ đề 2:** gọi `i` = độ dài tiền tố chung lớn nhất (≤ `n`), `j` = độ dài hậu tố chung lớn nhất (≤ `n`). Khi đó `a[:k] = b[:k] ⇔ k ≤ i` và `a[m-n+k:] = b[k:] ⇔ n − k ≤ j`.

**Kết luận:** tồn tại `k` ⇔ khoảng `[n − j, i] ∩ [0, n]` khác rỗng ⇔ `n − j ≤ i` ⇔ `i + j ≥ n` (vì `0 ≤ i, j ≤ n` nên giao với `[0, n]` tự động). Hai vòng `while` tính đúng `i`, `j`.

**Cách 4 (deque):** bóc trái tối đa được `i' = min(i, …)` — chính là `i` khi câu ngắn không bị bóc hết; nếu bị bóc hết thì đã đúng. Bóc phải trên phần còn lại đếm được `min(j, n − i)` từ; câu ngắn rỗng ⇔ `i + min(j, n − i) = n` ⇔ `i + j ≥ n`.

## 11. Phân tích Time / Space

Gọi `L = len(sentence1) + len(sentence2) ≤ 200`, `m`, `n` = số từ.

| Cách | Time | Space |
|---|---|---|
| 1. Xóa mọi khối | `O(m³)` | `O(m)` |
| 2. Mọi điểm cắt | `O(n²) + O(L)` | `O(m + n)` |
| 3. Hai con trỏ | `O(L)` | `O(m + n)` |
| 4. Deque | `O(L)` | `O(m + n)` |

(So sánh hai từ tốn `O(độ dài từ)`, tổng vẫn bị chặn bởi `O(L)`.) Dù ràng buộc nhỏ, ta đo trên câu dài để thấy khác biệt:

In [14]:
rng = random.Random(1813)
words = [rng.choice(["a", "b"]) for _ in range(400)]
s_long = " ".join(words)
s_short = " ".join(words[:150] + words[260:])
for name, f in [("Cách 1", similar_brute), ("Cách 2", similar_cuts),
                ("Cách 3", SolutionTwoPointers().areSentencesSimilar), ("Cách 4", SolutionDeque().areSentencesSimilar)]:
    t = time.perf_counter(); r = f(s_long, s_short); dt = time.perf_counter() - t
    print(f"{name}: {r} trong {1000*dt:.2f} ms")

Cách 1: True trong 42.01 ms
Cách 2: True trong 0.20 ms
Cách 3: True trong 0.03 ms
Cách 4: True trong 0.06 ms


## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [15]:
sols = [Solution(), SolutionTwoPointers(), SolutionDeque()]
edge = examples + [
    ("A", "A", True),                 # bằng nhau
    ("A", "B", False),
    ("A B", "B A", False),
    ("A B C", "A C", True),
    ("A A A", "A A", True),           # từ lặp: i và j chồng nhau
    ("A B", "A", True),
    ("B A", "A", True),               # chèn ở đầu
    ("C A B", "A B", True),
    ("a", "A", False),                # phân biệt hoa thường
    ("xx xx y", "xx y y", False),
    ("Eating", "Eating right now", True),  # đổi chỗ thứ tự tham số
]
for a, b, want in edge:
    for s in sols:
        assert s.areSentencesSimilar(a, b) == want, (a, b, s)
    assert similar_brute(a, b) == want == similar_cuts(a, b)
    print("OK", repr(a), repr(b), "->", want)

OK 'My name is Haley' 'My Haley' -> True
OK 'of' 'A lot of words' -> False
OK 'Eating right now' 'Eating' -> True
OK 'Hello Jane' 'Hello my name is Jane' -> True
OK 'Frog cool' 'Frogs are cool' -> False
OK 'A' 'A' -> True
OK 'A' 'B' -> False
OK 'A B' 'B A' -> False
OK 'A B C' 'A C' -> True
OK 'A A A' 'A A' -> True
OK 'A B' 'A' -> True
OK 'B A' 'A' -> True
OK 'C A B' 'A B' -> True
OK 'a' 'A' -> False
OK 'xx xx y' 'xx y y' -> False
OK 'Eating' 'Eating right now' -> True


### 12.2 Ngẫu nhiên có seed, đối chiếu vét cạn

Sinh câu từ bảng từ nhỏ `{A, B, C}` (dễ trùng, dễ có từ lặp); một nửa số case được tạo bằng cách **xóa một khối** khỏi câu dài (chắc chắn tương tự), nửa còn lại ngẫu nhiên hoàn toàn. Độ dài chuỗi ≤ 100 đúng ràng buộc.

In [16]:
def rand_sentence(rng, k):
    return " ".join(rng.choice(["A", "B", "C", "Ab"]) for _ in range(k))

rng = random.Random(7)
cnt_true = 0
for _ in range(3000):
    a = rand_sentence(rng, rng.randint(1, 12))
    if rng.random() < 0.5:
        w = a.split(); l = rng.randint(0, len(w)); r = rng.randint(l, len(w))
        b_words = w[:l] + w[r:]
        b = " ".join(b_words) if b_words else rng.choice(["A", "B"])
    else:
        b = rand_sentence(rng, rng.randint(1, 12))
    if rng.random() < 0.5:
        a, b = b, a
    assert len(a) <= 100 and len(b) <= 100
    want = similar_brute(a, b)
    cnt_true += want
    for s in sols:
        assert s.areSentencesSimilar(a, b) == want, (a, b)
    assert similar_cuts(a, b) == want
print("3000 case khớp vét cạn; số case true:", cnt_true)

3000 case khớp vét cạn; số case true: 1598


## 13. Lỗi người mới hay gặp

### Lỗi 1 — So ở mức ký tự (`startswith`/`endswith` trên chuỗi gốc)

In [17]:
def wrong_char_level(s1, s2):
    a, b = (s1, s2) if len(s1) >= len(s2) else (s2, s1)
    bw = b.split()
    for k in range(len(bw) + 1):
        pre, suf = " ".join(bw[:k]), " ".join(bw[k:])
        if a.startswith(pre) and a.endswith(suf):      # SAI: khớp được cả nửa từ
            return True
    return False

print("sai:", wrong_char_level("Frog cool", "Frogs are cool"), "| đúng:", Solution().areSentencesSimilar("Frog cool", "Frogs are cool"))

sai: True | đúng: False


**Vì sao sai:** `"Frogs are cool".startswith("Frog")` đúng ở mức ký tự, nhưng `"Frog"` không phải một **từ** của câu dài. Đề yêu cầu đoạn chèn phải ngăn cách bằng dấu cách. **Sửa:** `split()` rồi so từng từ.

### Lỗi 2 — Đòi `i + j == n` (không cho chồng nhau)

In [18]:
def wrong_equal(s1, s2):
    a, b = s1.split(), s2.split()
    if len(a) < len(b): a, b = b, a
    m, n = len(a), len(b)
    i = 0
    while i < n and a[i] == b[i]: i += 1
    j = 0
    while j < n and a[m - 1 - j] == b[n - 1 - j]: j += 1
    return i + j == n                                   # SAI

for a, b in [("A A", "A"), ("A A A", "A A")]:
    print(repr(a), repr(b), "sai:", wrong_equal(a, b), "| đúng:", Solution().areSentencesSimilar(a, b))

'A A' 'A' sai: False | đúng: True
'A A A' 'A A' sai: False | đúng: True


**Vì sao sai:** với từ lặp, tiền tố và hậu tố chung có thể **phủ chồng** lên nhau (`"A A"` vs `"A"`: `i = j = 1`, `i + j = 2 > 1`). Chỉ cần **tồn tại** một điểm cắt ⇒ `≥`.

### Lỗi 3 — Không đổi chỗ, dùng sai giới hạn vòng lặp

In [19]:
def wrong_no_swap(s1, s2):
    a, b = s1.split(), s2.split()
    m, n = len(a), len(b)
    i = 0
    while i < n and a[i] == b[i]:                       # SAI nếu a ngắn hơn b
        i += 1
    j = 0
    while j < n and a[m - 1 - j] == b[n - 1 - j]:
        j += 1
    return i + j >= n

try:
    print(wrong_no_swap("Eating", "Eating right now"))
except IndexError as e:
    print("IndexError:", e, "| đúng:", Solution().areSentencesSimilar("Eating", "Eating right now"))
print("trường hợp im lặng sai:", wrong_no_swap("A", "B A"), "| đúng:", Solution().areSentencesSimilar("A", "B A"))

IndexError: list index out of range | đúng: True
trường hợp im lặng sai: False | đúng: True


**Vì sao sai:** khi `sentence1` ngắn hơn, `a[i]` vượt chỉ số (lỗi rõ), hoặc tệ hơn: `a[m-1-j]` với chỉ số âm **không báo lỗi** mà quay vòng từ cuối list ⇒ so sai lặng lẽ. **Sửa:** đổi chỗ để `b` luôn là câu ngắn.

### Lỗi 4 — Chỉ cho chèn **ở giữa** (bắt buộc `i ≥ 1` và `j ≥ 1`)

In [20]:
def wrong_middle_only(s1, s2):
    a, b = s1.split(), s2.split()
    if len(a) < len(b): a, b = b, a
    m, n = len(a), len(b)
    i = 0
    while i < n and a[i] == b[i]: i += 1
    j = 0
    while j < n and a[m - 1 - j] == b[n - 1 - j]: j += 1
    return i >= 1 and j >= 1 and i + j >= n             # SAI: cấm chèn đầu/cuối

print("sai:", wrong_middle_only("Eating right now", "Eating"), "| đúng:", Solution().areSentencesSimilar("Eating right now", "Eating"))

sai: False | đúng: True


**Vì sao sai:** "chèn vào bên trong" bao gồm cả vị trí **đầu** và **cuối** câu (Ví dụ 3 chèn ở cuối). **Sửa:** bỏ điều kiện `i ≥ 1`, `j ≥ 1`.

### Lỗi 5 — Kiểm "câu ngắn là dãy con của câu dài"

In [21]:
def wrong_subsequence(s1, s2):
    a, b = s1.split(), s2.split()
    if len(a) < len(b): a, b = b, a
    it = iter(a)
    return all(w in it for w in b)                      # SAI: cho phép chèn nhiều chỗ

print("sai:", wrong_subsequence("of", "A lot of words"), "| đúng:", Solution().areSentencesSimilar("of", "A lot of words"))

sai: True | đúng: False


**Vì sao sai:** "dãy con" cho phép chèn ở **nhiều** chỗ (ở đây cần chèn `"A lot"` trước và `"words"` sau). Đề chỉ cho chèn **một** câu ở **một** chỗ.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Chèn/xóa đúng một khối liên tiếp"** ⇒ phần còn lại = tiền tố chung + hậu tố chung ⇒ hai con trỏ từ hai đầu, điều kiện `i + j ≥ n`.
- Làm việc với **token** (từ) khi đề nói về "câu", "từ", "phân cách bằng dấu cách".
- Đổi chỗ để chuẩn hóa (câu ngắn/dài) trước khi viết vòng lặp.
- Cẩn thận chỉ số âm của Python: lỗi lệch biên có thể **không** ném ngoại lệ.
- Test nhanh: chèn đầu, chèn cuối, câu bằng nhau, từ lặp, khác hoa/thường.

## 15. Biến thể của bài

1. **LeetCode 734 / 737 — Sentence Similarity I/II:** hai câu cùng độ dài, từ tương ứng phải "tương tự" theo danh sách cặp (II có bắc cầu ⇒ Union-Find).
2. **Cho phép chèn tối đa `k` đoạn** (Bài tập 5) ⇒ DP trên vị trí hai câu + số đoạn đã dùng.
3. **Ở mức ký tự** (chèn một chuỗi con bất kỳ): `t` thu được từ `s` bằng xóa một khối ⇔ tiền tố chung + hậu tố chung ≥ `len(t)` — cùng công thức.
4. **Trả về đoạn cần chèn** (Bài tập 3).
5. Bài cùng mẫu hai đầu: LeetCode 1750 *Minimum Length of String After Deleting Similar Ends*, 2565 *Subsequence With the Minimum Score*.

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Xóa mọi khối | thử mọi `(l, r)` | `O(m³)` | `O(m)` | chuẩn đối chiếu |
| 2. Mọi điểm cắt | thử `k = 0..n` | `O(n²)` | `O(m+n)` | dễ hiểu, `n` nhỏ |
| 3. Hai con trỏ | `i + j ≥ n` | `O(L)` | `O(m+n)` | **nộp bài** |
| 4. Deque | bóc hai đầu | `O(L)` | `O(m+n)` | không cần đổi chỗ |

## 17. 5 bài tập từ dễ đến khó

**Bài 1 (đoán output):** `("A B C", "A C")`, `("A B", "B A")`, `("c h p Ny", "c BDQ r h p Ny")`?

**Bài 2 (tính tay):** Ghi `i`, `j`, `n` cho Ví dụ 1 và Ví dụ 3.

**Bài 3 (điền TODO):** Viết `insertion(s1, s2)` trả về `(vị_trí_chèn_k, đoạn_chèn)` (đoạn chèn là chuỗi các từ, có thể rỗng) nếu tương tự, ngược lại `None`. Nếu có nhiều cách, trả về `k` nhỏ nhất.

**Bài 4 (tự viết lại):** Viết `similar_chars(s, t)` — phiên bản **mức ký tự**: có thể chèn một chuỗi con bất kỳ (có thể rỗng) vào chuỗi ngắn để được chuỗi dài không?

**Bài 5 (mini challenge):** Viết `similar_k(s1, s2, k)` — cho phép chèn **tối đa `k` đoạn** (mỗi đoạn ở một chỗ) vào câu ngắn để được câu dài.

In [22]:
def insertion(s1, s2):
    # TODO
    pass

def similar_chars(s, t):
    # TODO
    pass

def similar_k(s1, s2, k):
    # TODO: DP
    pass

## 18. Gợi ý 3 mức cho bài tập

| Bài | Gợi ý 1 | Gợi ý 2 | Gợi ý 3 |
|---|---|---|---|
| 1 | Tính `i`, `j` | Câu 2: `A ≠ B` ở đầu, `B ≠ A` ở cuối | Câu 3: `i = 1` (`c`), `j = 2` (`h p Ny`) |
| 2 | VD1: `n = 2` | `i = 1`, `j = 1` | VD3: `i = 1`, `j = 0`, `n = 1` |
| 3 | Tính `i`, `j` | `k` nhỏ nhất = `max(0, n − j)` | Đoạn chèn `a[k : m − n + k]` |
| 4 | Giống hệt bài nhưng trên ký tự | `i`, `j` trên chuỗi | `i + j ≥ len(ngắn)` |
| 5 | `dp[p][q][g]`: đã khớp `p` từ của câu dài, `q` từ câu ngắn, dùng `g` đoạn | Bước: khớp từ (`p+1, q+1`) hoặc bắt đầu/tiếp tục một đoạn chèn | Đánh dấu "đang trong đoạn chèn" để không đếm đoạn hai lần |

## 19. Đáp án tham khảo

In [23]:
r1 = [Solution().areSentencesSimilar(a, b) for a, b in [("A B C", "A C"), ("A B", "B A"), ("c h p Ny", "c BDQ r h p Ny")]]
print("Bài 1:", r1); assert r1 == [True, False, True]
print("Bài 2:"); trace_two_pointers("My name is Haley", "My Haley"); trace_two_pointers("Eating right now", "Eating")

Bài 1: [True, False, True]
Bài 2:
  tiền tố: a[0]='My' == b[0]='My'
  tiền tố dừng: a[1]='name' != b[1]='Haley'
  hậu tố: a[3]='Haley' == b[1]='Haley'
  hậu tố dừng: a[2]='is' != b[0]='My'
  i=1, j=1, n=2 -> True
  tiền tố: a[0]='Eating' == b[0]='Eating'
  hậu tố dừng: a[2]='now' != b[0]='Eating'
  i=1, j=0, n=1 -> True


True

In [24]:
def insertion(s1, s2):
    a, b = s1.split(), s2.split()
    if len(a) < len(b): a, b = b, a
    m, n = len(a), len(b)
    i = 0
    while i < n and a[i] == b[i]: i += 1
    j = 0
    while j < n and a[m - 1 - j] == b[n - 1 - j]: j += 1
    if i + j < n:
        return None
    k = max(0, n - j)
    return k, " ".join(a[k:m - n + k])

def similar_chars(s, t):
    if len(s) < len(t): s, t = t, s
    m, n = len(s), len(t)
    i = 0
    while i < n and s[i] == t[i]: i += 1
    j = 0
    while j < n and s[m - 1 - j] == t[n - 1 - j]: j += 1
    return i + j >= n

def similar_chars_brute(s, t):
    if len(s) < len(t): s, t = t, s
    return any(s[:l] + s[r:] == t for l in range(len(s) + 1) for r in range(l, len(s) + 1))

print("Bài 3:", insertion("My name is Haley", "My Haley"), insertion("Eating", "Eating right now"), insertion("of", "A lot of words"))
rng = random.Random(3)
for _ in range(2000):
    a = rand_sentence(rng, rng.randint(1, 8)); b = rand_sentence(rng, rng.randint(1, 8))
    res = insertion(a, b)
    assert (res is not None) == Solution().areSentencesSimilar(a, b)
    if res is not None:
        k, ins = res
        aw, bw = (a.split(), b.split()) if len(a.split()) >= len(b.split()) else (b.split(), a.split())
        assert bw[:k] + ins.split() + bw[k:] == aw
    x = "".join(rng.choice("ab") for _ in range(rng.randint(0, 7)))
    y = "".join(rng.choice("ab") for _ in range(rng.randint(0, 7)))
    assert similar_chars(x, y) == similar_chars_brute(x, y)
print("Bài 3, 4: OK")

Bài 3: (1, 'name is') (1, 'right now') None
Bài 3, 4: OK


In [25]:
from functools import lru_cache
from itertools import combinations

def similar_k(s1, s2, k):
    a, b = s1.split(), s2.split()
    if len(a) < len(b): a, b = b, a
    m, n = len(a), len(b)

    @lru_cache(maxsize=None)
    def go(p, q, g, inside):
        # p: số từ câu dài đã xử lý, q: số từ câu ngắn đã khớp, g: số đoạn chèn đã mở
        if p == m:
            return q == n
        res = False
        if q < n and a[p] == b[q]:
            res = go(p + 1, q + 1, g, False)               # khớp từ, đóng đoạn chèn (nếu đang mở)
        if not res:
            if inside:
                res = go(p + 1, q, g, True)                # kéo dài đoạn chèn hiện tại
            elif g < k:
                res = go(p + 1, q, g + 1, True)            # mở đoạn chèn mới
        return res

    return go(0, 0, 0, False)

def similar_k_brute(s1, s2, k):
    a, b = s1.split(), s2.split()
    if len(a) < len(b): a, b = b, a
    m, n = len(a), len(b)
    # chọn tập vị trí của a để giữ lại (khớp b), kiểm số khối bị xóa <= k
    for keep in combinations(range(m), n):
        if [a[t] for t in keep] != b:
            continue
        removed = [t not in set(keep) for t in range(m)]
        blocks = sum(1 for t in range(m) if removed[t] and (t == 0 or not removed[t - 1]))
        if blocks <= k:
            return True
    return False

assert similar_k("of", "A lot of words", 1) is False and similar_k("of", "A lot of words", 2) is True
rng = random.Random(11)
for _ in range(600):
    a = rand_sentence(rng, rng.randint(1, 8)); b = rand_sentence(rng, rng.randint(1, 6)); k = rng.randint(0, 3)
    assert similar_k(a, b, k) == similar_k_brute(a, b, k), (a, b, k)
    assert similar_k(a, b, 1) == Solution().areSentencesSimilar(a, b)
print("Bài 5 (biến thể tự dựng — đối chiếu lại đề gốc nếu có): OK")

Bài 5 (biến thể tự dựng — đối chiếu lại đề gốc nếu có): OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
a, b = s1.split(), s2.split()
if len(a) < len(b): a, b = b, a        # b là câu ngắn
m, n = len(a), len(b)
i = 0
while i < n and a[i] == b[i]: i += 1                      # tiền tố chung
j = 0
while j < n and a[m-1-j] == b[n-1-j]: j += 1              # hậu tố chung
return i + j >= n
```

**Checklist:**
- [ ] So theo **từ** (`split()`), không theo ký tự?
- [ ] Đổi chỗ để `b` là câu ngắn (tránh chỉ số âm âm thầm)?
- [ ] Điều kiện `i + j >= n` (cho phép chồng nhau)?
- [ ] Cho phép chèn ở đầu/cuối (`i = 0` hoặc `j = 0` vẫn được)?
- [ ] Phân biệt hoa/thường (`"a" != "A"`).